In [ ]:
# ─── CELL 1.1 | Mount Google Drive ───────────────────────────────────────────
# Run this first. A popup will ask for permission — allow it.
# All datasets and models will be saved to your Drive so you don't lose them.

from google.colab import drive
drive.mount('/content/drive')

import os

# Create your thesis project folder inside Drive
PROJECT_DIR = '/content/drive/MyDrive/Pneumonia_Thesis'
os.makedirs(PROJECT_DIR, exist_ok=True)

print(f"Project folder ready: {PROJECT_DIR}")

In [ ]:
# ─── CELL 1.2 | Install libraries (FIXED) ────────────────────────────────────

!pip install -q \
    opencv-python-headless \
    scikit-learn \
    scikit-image \
    matplotlib \
    seaborn \
    shap \
    tqdm \
    Pillow \
    kaggle

# TensorFlow comes pre-installed on Colab — just verify the version
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print("All libraries installed.")

In [ ]:
# ─── CELL 1.3 | Kaggle API setup (NEW TOKEN FORMAT) ──────────────────────────
# Paste your token string below where it says YOUR_TOKEN_HERE
# Example: KGAT_31e8ab18e3087afb300e243ee912b1a7

import os

KAGGLE_TOKEN = "KGAT_31e8ab18e3087afb300e243ee912b1a7"   # ← paste your token here

# Save it where the Kaggle CLI expects it
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
token_path = os.path.expanduser('~/.kaggle/access_token')

with open(token_path, 'w') as f:
    f.write(KAGGLE_TOKEN)
os.chmod(token_path, 0o600)

# Verify it works
result = os.popen('kaggle datasets list').read()
print(result[:500] if result else "Something went wrong — check your token")

In [ ]:
# ─── FIX CELL | Re-extract dataset from zip ──────────────────────────────────

import os
import zipfile
import shutil

PROJECT_DIR = '/content/drive/MyDrive/Pneumonia_Thesis'
KAGGLE_DIR  = os.path.join(PROJECT_DIR, 'chest_xray')
ZIP_PATH    = os.path.join(PROJECT_DIR, 'chest-xray-pneumonia.zip')

# Delete the bad extraction
shutil.rmtree(KAGGLE_DIR)
os.makedirs(KAGGLE_DIR, exist_ok=True)

# Extract manually
print("Extracting zip file... (may take 1-2 minutes)")
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(KAGGLE_DIR)
print("Extraction done.")

# See what came out
print("\nContents after extraction:")
for item in os.listdir(KAGGLE_DIR):
    full = os.path.join(KAGGLE_DIR, item)
    kind = "DIR" if os.path.isdir(full) else "FILE"
    print(f"  [{kind}] {item}")

In [ ]:
# ─── CELL 1.4 | Download Kaggle Chest X-Ray dataset ──────────────────────────
# Dataset: chest-xray-pneumonia by Paul Mooney
# Size: ~1.2 GB  |  5,216 train + 624 test images

KAGGLE_DIR = os.path.join(PROJECT_DIR, 'chest_xray')

if not os.path.exists(KAGGLE_DIR):
    print("Downloading Kaggle CXR dataset (~1.2 GB) ...")
    !kaggle datasets download -d paultimothymooney/chest-xray-pneumonia \
        -p "{PROJECT_DIR}" --unzip
    print("Download complete.")
else:
    print("Kaggle CXR dataset already exists. Skipping download.")

# Verify structure
for split in ['train', 'val', 'test']:
    for cls in ['NORMAL', 'PNEUMONIA']:
        path = os.path.join(KAGGLE_DIR, split, cls)
        count = len(os.listdir(path)) if os.path.exists(path) else 0
        print(f"  {split:5s} / {cls:10s} → {count:5d} images")

Kaggle CXR dataset already exists. Skipping download.
  train / NORMAL     →     0 images
  train / PNEUMONIA  →     0 images
  val   / NORMAL     →     0 images
  val   / PNEUMONIA  →     0 images
  test  / NORMAL     →     0 images
  test  / PNEUMONIA  →     0 images


In [ ]:
# ─── CELL 1.5 | Download Montgomery County dataset ───────────────────────────
# Dataset: montgomery-county-chest-x-ray
# Contains 138 CXR images with LEFT and RIGHT lung masks (ground truth)
# This is what we use to train the U-Net segmentation model

MONTGOMERY_DIR = os.path.join(PROJECT_DIR, 'montgomery')

if not os.path.exists(MONTGOMERY_DIR):
    print("Downloading Montgomery County dataset ...")
    !kaggle datasets download -d raddar/montgomery-county-chest-x-ray \
        -p "{PROJECT_DIR}" --unzip
    # Rename folder for consistency
    for name in os.listdir(PROJECT_DIR):
        full = os.path.join(PROJECT_DIR, name)
        if os.path.isdir(full) and 'montgomery' in name.lower() and full != MONTGOMERY_DIR:
            os.rename(full, MONTGOMERY_DIR)
            break
    print("Download complete.")
else:
    print("Montgomery dataset already exists. Skipping download.")

# Verify — should show CXR_png folder and masks
print("\nMontgomery folder contents:")
for item in sorted(os.listdir(MONTGOMERY_DIR))[:10]:
    print(f"  {item}")